# 🔬 Lab Module 17 — Multi-Modal Agents & Document Intelligence

**Mục tiêu:** Xây dựng hoàn chỉnh Document Intelligence pipeline cho LoanBot v3.3

**Các thành phần:**
1. Mock document data (simulate Claude Vision extraction)
2. Document Classifier
3. Confidence-based validation
4. Cross-check engine
5. Collateral verification chain
6. Full pipeline orchestration
7. 🎯 Practice: Tamper detection

---
**Context:** FinTech Corp xử lý 50,000 hồ sơ/tháng — mỗi hồ sơ kèm 5–12 documents.
LoanBot v3.3 phải extract, validate, cross-check toàn bộ trong <90 giây.

## Section 1: Setup & Mock Data

Trong lab này chúng ta dùng **mock data** để simulate Claude Vision extraction. Trong production, `extract_from_document()` gọi Claude API thật.

In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Any
import json, time, asyncio, hashlib
from enum import Enum

# ─── Loan Applications (4 test customers) ────────────────────────────────────
APPLICATIONS = {
    "FC-2024-001": {
        "name": "Nguyễn Văn An",
        "cccd": "001234567890",
        "declared_income": 45_000_000,   # VNĐ/tháng
        "loan_amount": 800_000_000,
        "loan_purpose": "home_purchase",
        "collateral_address": "123 Lê Lợi, Quận 1, TP.HCM",
    },
    "FC-2024-002": {
        "name": "Trần Thị Bình",
        "cccd": "001234567891",
        "declared_income": 28_000_000,
        "loan_amount": 300_000_000,
        "loan_purpose": "personal",
        "collateral_address": None,  # unsecured
    },
    "FC-2024-003": {
        "name": "Lê Minh Cường",
        "cccd": "001234567892",
        "declared_income": 120_000_000,  # claimed — suspicious
        "loan_amount": 2_000_000_000,
        "loan_purpose": "business",
        "collateral_address": "456 Nguyễn Trãi, Quận 5, TP.HCM",
    },
    "FC-2024-004": {
        "name": "Phạm Thị Dung",
        "cccd": "001234567893",
        "declared_income": 32_000_000,
        "loan_amount": 500_000_000,
        "loan_purpose": "home_purchase",
        "collateral_address": "789 Trần Hưng Đạo, Quận 5, TP.HCM",
    }
}

print("✅ Applications loaded:", list(APPLICATIONS.keys()))

In [ ]:
# ─── Mock Document Extractions ────────────────────────────────────────────────
# Simulate kết quả từ Claude Vision API cho từng document type

MOCK_DOCUMENTS = {
    "FC-2024-001": {
        "cccd": {
            "type": "cccd",
            "full_name": "Nguyễn Văn An",
            "id_number": "001234567890",
            "dob": "1985-03-15",
            "issue_date": "2022-01-10",
            "expiry_date": "2032-01-10",
            "confidence": 0.99
        },
        "payslip": {
            "type": "payslip",
            "employee_name": "Nguyễn Văn An",
            "employer": "Vinhomes JSC",
            "id_number": "001234567890",
            "period": "2026-05",
            "gross_income": 52_000_000,
            "net_income": 45_500_000,
            "total_deductions": 6_500_000,
            "allowances": 2_000_000,
            "confidence": 0.97
        },
        "bank_statement": {
            "type": "bank_statement",
            "account_holder": "Nguyễn Văn An",
            "bank": "Vietcombank",
            "period_months": 12,
            "avg_monthly_inflow": 44_800_000,
            "avg_monthly_outflow": 30_200_000,
            "avg_end_balance": 18_500_000,
            "overdraft_count": 0,
            "income_source": "employment",
            "income_regularity": "regular",
            "employer_name": "Vinhomes JSC",
            "red_flags": [],
            "confidence": 0.96
        },
        "property_deed": {
            "type": "property_deed",
            "certificate_number": "BH-123456",
            "issue_date": "2019-06-01",
            "issuing_authority": "UBND Quận 1",
            "owner_names": ["Nguyễn Văn An"],
            "owner_ids": ["001234567890"],
            "parcel_id": "Q1-123-456",
            "address": "123 Lê Lợi, Quận 1, TP.HCM",
            "area_sqm": 85.0,
            "land_purpose": "residential",
            "land_use_term_years": None,  # freehold
            "encumbrances": [],
            "has_existing_mortgage": False,
            "confidence": 0.95
        }
    },
    "FC-2024-003": {  # Suspicious customer
        "payslip": {
            "type": "payslip",
            "employee_name": "Lê Minh Cường",
            "employer": "ABC Trading Co",
            "id_number": "001234567892",
            "period": "2026-05",
            "gross_income": 120_000_000,   # suspicious round number
            "net_income": 120_000_000,     # no deductions?? red flag
            "total_deductions": 0,         # impossible
            "allowances": 0,
            "confidence": 0.61             # low confidence — blurry scan
        },
        "property_deed": {
            "type": "property_deed",
            "certificate_number": "BH-999999",
            "owner_names": ["Lê Văn Khác"],  # not the applicant!
            "owner_ids": ["999888777666"],
            "address": "456 Nguyễn Trãi, Quận 5, TP.HCM",
            "area_sqm": 200.0,
            "has_existing_mortgage": True,   # already mortgaged
            "encumbrances": ["Thế chấp tại Techcombank — hợp đồng số TCB-2025-001"],
            "confidence": 0.91
        }
    },
    "FC-2024-004": {
        "cccd": {"type": "cccd", "full_name": "Phạm Thị Dung", "id_number": "001234567893", "confidence": 0.99},
        "payslip": {
            "type": "payslip", "employee_name": "Phạm Thị Dung", "employer": "FPT Software",
            "period": "2026-05", "gross_income": 36_000_000, "net_income": 31_500_000,
            "total_deductions": 4_500_000, "confidence": 0.97
        },
        "bank_statement": {
            "type": "bank_statement", "account_holder": "Phạm Thị Dung",
            "bank": "VCB", "period_months": 12,
            "avg_monthly_inflow": 31_250_000, "avg_monthly_outflow": 22_000_000,
            "avg_end_balance": 8_400_000, "overdraft_count": 0,
            "income_source": "employment", "income_regularity": "regular",
            "employer_name": "FPT Software", "red_flags": [], "confidence": 0.94
        },
        "property_deed": {  # Deed in husband's name
            "type": "property_deed",
            "owner_names": ["Nguyễn Văn Bình"],  # husband's name
            "owner_ids": ["001111222333"],
            "address": "789 Trần Hưng Đạo, Quận 5, TP.HCM",
            "area_sqm": 65.0,
            "has_existing_mortgage": False,
            "encumbrances": [],
            "confidence": 0.96
        }
    }
}

print("✅ Mock documents loaded for:", list(MOCK_DOCUMENTS.keys()))

## Section 2: Document Classifier & Confidence Validator

In [ ]:
class ConfidenceLevel(Enum):
    AUTO_ACCEPT    = "auto_accept"    # >= 0.95
    FLAG_REVIEW    = "flag_review"    # 0.80 - 0.95
    REQUEST_REUPLOAD = "request_reupload"  # 0.60 - 0.80
    HITL_MANDATORY = "hitl_mandatory"  # < 0.60

def classify_confidence(score: float) -> ConfidenceLevel:
    """Per-field confidence classification."""
    if score >= 0.95:   return ConfidenceLevel.AUTO_ACCEPT
    if score >= 0.80:   return ConfidenceLevel.FLAG_REVIEW
    if score >= 0.60:   return ConfidenceLevel.REQUEST_REUPLOAD
    return ConfidenceLevel.HITL_MANDATORY

@dataclass
class ValidationResult:
    doc_type:    str
    confidence:  float
    level:       ConfidenceLevel
    issues:      List[str] = field(default_factory=list)
    warnings:    List[str] = field(default_factory=list)
    hitl_required: bool = False
    red_flags:   List[str] = field(default_factory=list)

def validate_document(doc: Dict[str, Any]) -> ValidationResult:
    """Validate document confidence + detect red flags."""
    doc_type   = doc.get("type", "unknown")
    confidence = doc.get("confidence", 0.0)
    level      = classify_confidence(confidence)
    issues, warnings, red_flags = [], [], []
    hitl = level == ConfidenceLevel.HITL_MANDATORY

    # Doc-specific validation
    if doc_type == "payslip":
        # Impossible: net > gross
        if doc.get("net_income", 0) > doc.get("gross_income", 0):
            issues.append("net_income > gross_income — impossible")
            hitl = True
        # Suspicious: zero deductions
        if doc.get("total_deductions", -1) == 0 and doc.get("gross_income", 0) > 10_000_000:
            red_flags.append("Zero deductions on high income — potential tampering")
        # Suspicious: round number income
        net = doc.get("net_income", 0)
        if net > 0 and net % 10_000_000 == 0:
            red_flags.append(f"Round number net_income={net:,} VNĐ — verify authenticity")

    elif doc_type == "bank_statement":
        overdrafts = doc.get("overdraft_count", 0)
        if overdrafts > 2:
            red_flags.append(f"Overdraft {overdrafts} times — financial stress signal")
        if doc.get("red_flags"):
            red_flags.extend(doc["red_flags"])

    elif doc_type == "property_deed":
        if doc.get("has_existing_mortgage"):
            issues.append("Existing mortgage on property — cannot use as collateral")
            hitl = True

    if level == ConfidenceLevel.FLAG_REVIEW:
        warnings.append(f"Low confidence ({confidence:.2f}) — loan officer review recommended")
    elif level == ConfidenceLevel.REQUEST_REUPLOAD:
        issues.append(f"Very low confidence ({confidence:.2f}) — request document re-upload")
        hitl = True

    return ValidationResult(
        doc_type=doc_type, confidence=confidence, level=level,
        issues=issues, warnings=warnings, hitl_required=hitl, red_flags=red_flags
    )

# Test: validate FC-2024-003 payslip (suspicious)
r = validate_document(MOCK_DOCUMENTS["FC-2024-003"]["payslip"])
print(f"FC-2024-003 payslip validation:")
print(f"  confidence:    {r.confidence}")
print(f"  level:         {r.level.value}")
print(f"  issues:        {r.issues}")
print(f"  red_flags:     {r.red_flags}")
print(f"  hitl_required: {r.hitl_required}")

## Section 3: Cross-Check Engine

In [ ]:
@dataclass
class CrossCheckResult:
    income_verified:         bool
    income_discrepancy_pct:  float
    name_consistent:         bool
    id_consistent:           bool
    employer_consistent:     bool  # payslip employer matches bank statement
    discrepancies:           List[str] = field(default_factory=list)
    hitl_required:           bool = False

def cross_check_documents(app_id: str, docs: Dict[str, Dict]) -> CrossCheckResult:
    """Compare extracted document data against declared application data."""
    app      = APPLICATIONS[app_id]
    payslip  = docs.get("payslip", {})
    bankstmt = docs.get("bank_statement", {})
    cccd     = docs.get("cccd", {})
    discrepancies = []
    hitl = False

    # Income: compare bank avg_inflow vs declared
    declared = app["declared_income"]
    bank_inflow = bankstmt.get("avg_monthly_inflow", 0)
    disc_pct = abs(bank_inflow - declared) / declared if declared > 0 else 1.0
    income_verified = disc_pct < 0.10
    if disc_pct >= 0.20:
        discrepancies.append(f"Income discrepancy {disc_pct:.1%}: declared={declared:,} vs bank_avg={bank_inflow:,} VNĐ")
        hitl = True

    # Name consistency
    app_name = app["name"].lower()
    pay_name = payslip.get("employee_name", "").lower()
    ccc_name = cccd.get("full_name", "").lower()
    name_ok  = (not pay_name or app_name == pay_name) and (not ccc_name or app_name == ccc_name)
    if not name_ok:
        discrepancies.append(f"Name mismatch: app='{app_name}' payslip='{pay_name}' cccd='{ccc_name}'")
        hitl = True

    # ID consistency
    app_id_num = app["cccd"]
    pay_id_num = payslip.get("id_number", "")
    ccc_id_num = cccd.get("id_number", "")
    id_ok = (
        (not pay_id_num or app_id_num == pay_id_num) and
        (not ccc_id_num or app_id_num == ccc_id_num)
    )
    if not id_ok:
        discrepancies.append(f"ID mismatch: app={app_id_num} payslip={pay_id_num} cccd={ccc_id_num}")
        hitl = True

    # Employer consistency payslip vs bank
    pay_employer  = payslip.get("employer", "").lower()
    bank_employer = bankstmt.get("employer_name", "").lower()
    emp_ok = not pay_employer or not bank_employer or pay_employer in bank_employer or bank_employer in pay_employer
    if not emp_ok:
        discrepancies.append(f"Employer mismatch: payslip='{pay_employer}' vs bank='{bank_employer}'")

    return CrossCheckResult(
        income_verified=income_verified, income_discrepancy_pct=disc_pct * 100,
        name_consistent=name_ok, id_consistent=id_ok, employer_consistent=emp_ok,
        discrepancies=discrepancies, hitl_required=hitl
    )

# Test FC-2024-004 (expect: income verified, no discrepancies)
cc4 = cross_check_documents("FC-2024-004", MOCK_DOCUMENTS["FC-2024-004"])
print("FC-2024-004 Cross-Check:")
print(f"  income_verified:      {cc4.income_verified} ({cc4.income_discrepancy_pct:.2f}% discrepancy)")
print(f"  name_consistent:      {cc4.name_consistent}")
print(f"  id_consistent:        {cc4.id_consistent}")
print(f"  discrepancies:        {cc4.discrepancies}")
print()

# Test FC-2024-003 (expect: income discrepancy, suspicious)
cc3 = cross_check_documents("FC-2024-003", MOCK_DOCUMENTS["FC-2024-003"])
print("FC-2024-003 Cross-Check:")
print(f"  income_discrepancy_pct: {cc3.income_discrepancy_pct:.1f}%")
print(f"  discrepancies:          {cc3.discrepancies}")
print(f"  hitl_required:          {cc3.hitl_required}")

## Section 4: Collateral Verification Chain

In [ ]:
@dataclass
class CollateralResult:
    chain_valid:      bool
    hitl_required:    bool
    hitl_level:       Optional[int]  # 1 = advisory, 2 = mandatory
    critical_issues:  List[str] = field(default_factory=list)
    warnings:         List[str] = field(default_factory=list)
    hitl_reason:      Optional[str] = None

# Mock valuation data (simulates PropertyValuation API tool result)
PROPERTY_VALUATIONS = {
    "123 Lê Lợi, Quận 1, TP.HCM":      {"address": "123 Lê Lợi, Quận 1, TP.HCM",   "area_sqm": 85.0, "value_vnd": 8_500_000_000},
    "456 Nguyễn Trãi, Quận 5, TP.HCM": {"address": "456 Nguyễn Trãi, Quận 5, TP.HCM","area_sqm": 198.0,"value_vnd": 15_000_000_000},
    "789 Trần Hưng Đạo, Quận 5, TP.HCM":{"address": "789 Trần Hưng Đạo, Quận 5, TP.HCM","area_sqm": 65.0,"value_vnd": 4_200_000_000},
}

def verify_collateral_chain(app_id: str, docs: Dict, valuations: Dict) -> CollateralResult:
    """Verify all 4 links: Identity + Encumbrance + Address + Area."""
    app  = APPLICATIONS[app_id]
    deed = docs.get("property_deed")

    if not deed:
        if app.get("loan_purpose") == "home_purchase" and app.get("collateral_address"):
            return CollateralResult(False, True, 2, ["Property deed missing for secured loan"])
        return CollateralResult(True, False, None, [], ["No collateral required"])

    collateral_addr = app.get("collateral_address", "")
    valuation = valuations.get(collateral_addr, {})
    issues, warnings = [], []

    # Link 1: Identity
    app_name    = app["name"].lower()
    deed_owners = [n.lower() for n in deed.get("owner_names", [])]
    identity_ok = any(app_name in o or o in app_name for o in deed_owners)
    if not identity_ok:
        # Check if it might be spousal property
        issues.append(f"Identity mismatch: '{app['name']}' not in deed owners {deed.get('owner_names', [])}")

    # Link 2: Encumbrance
    if deed.get("has_existing_mortgage"):
        issues.append("Existing mortgage — property cannot be used as collateral")
    if deed.get("encumbrances"):
        for enc in deed["encumbrances"]:
            issues.append(f"Encumbrance: {enc}")

    # Link 3: Address match
    deed_addr = deed.get("address", "").lower()[:40]
    val_addr  = valuation.get("address", "").lower()[:40]
    if deed_addr and val_addr and deed_addr != val_addr:
        warnings.append(f"Address mismatch: deed='{deed_addr}' vs valuation='{val_addr}'")

    # Link 4: Area match (5% tolerance)
    deed_area = deed.get("area_sqm", 0)
    val_area  = valuation.get("area_sqm", 0)
    if deed_area and val_area:
        area_diff = abs(deed_area - val_area) / deed_area
        if area_diff > 0.05:
            warnings.append(f"Area mismatch: deed={deed_area}m² vs valuation={val_area}m² ({area_diff:.1%})")

    chain_valid   = len(issues) == 0
    hitl_required = not chain_valid or len(warnings) > 0
    hitl_level    = 2 if issues else (1 if warnings else None)
    hitl_reason   = None
    if issues and any("Identity" in i for i in issues):
        hitl_reason = "owner_mismatch_spousal_property"
    elif issues and any("mortgage" in i for i in issues):
        hitl_reason = "existing_mortgage_on_collateral"

    return CollateralResult(
        chain_valid=chain_valid, hitl_required=hitl_required, hitl_level=hitl_level,
        critical_issues=issues, warnings=warnings, hitl_reason=hitl_reason
    )

# Test: FC-2024-001 (clean case)
col1 = verify_collateral_chain("FC-2024-001", MOCK_DOCUMENTS["FC-2024-001"], PROPERTY_VALUATIONS)
print(f"FC-2024-001 Collateral: chain_valid={col1.chain_valid}, hitl_level={col1.hitl_level}")
print()

# Test: FC-2024-003 (existing mortgage + identity mismatch)
col3 = verify_collateral_chain("FC-2024-003", MOCK_DOCUMENTS["FC-2024-003"], PROPERTY_VALUATIONS)
print(f"FC-2024-003 Collateral: chain_valid={col3.chain_valid}")
print(f"  critical_issues: {col3.critical_issues}")
print(f"  hitl_level:      {col3.hitl_level}")
print()

# Test: FC-2024-004 (deed in husband's name — spousal case)
col4 = verify_collateral_chain("FC-2024-004", MOCK_DOCUMENTS["FC-2024-004"], PROPERTY_VALUATIONS)
print(f"FC-2024-004 Collateral: chain_valid={col4.chain_valid}")
print(f"  critical_issues: {col4.critical_issues}")
print(f"  hitl_reason:     {col4.hitl_reason}")

## Section 5: Full Document Intelligence Pipeline

In [ ]:
class HITLLevel(Enum):
    NONE   = "none"
    L1     = "advisory"      # loan officer notified, can proceed
    L2     = "mandatory"     # must be reviewed before decision

@dataclass
class DocumentIntelligenceResult:
    app_id:            str
    overall_quality:   float      # avg confidence across all docs
    documents_ok:      List[str]  # auto-accepted docs
    documents_flagged: List[str]  # flagged docs
    documents_failed:  List[str]  # HITL mandatory docs
    cross_check:       Optional[CrossCheckResult]
    collateral:        Optional[CollateralResult]
    final_hitl_level:  HITLLevel
    hitl_reasons:      List[str]
    processing_time_s: float

def run_document_intelligence_pipeline(app_id: str) -> DocumentIntelligenceResult:
    """Run full Document Intelligence pipeline for a loan application."""
    t0   = time.time()
    docs = MOCK_DOCUMENTS.get(app_id, {})

    # Stage 1-3: validate each document
    docs_ok, docs_flagged, docs_failed = [], [], []
    all_confidences = []
    all_red_flags   = []
    hitl_reasons    = []

    for doc_name, doc_data in docs.items():
        vr = validate_document(doc_data)
        all_confidences.append(vr.confidence)
        all_red_flags.extend(vr.red_flags)

        if vr.hitl_required or vr.issues:
            docs_failed.append(doc_name)
            hitl_reasons.extend([f"{doc_name}: {i}" for i in (vr.issues or [f"confidence={vr.confidence:.2f}"])])
        elif vr.level == ConfidenceLevel.FLAG_REVIEW:
            docs_flagged.append(doc_name)
        else:
            docs_ok.append(doc_name)

    # Cross-check
    cc = None
    if len(docs) >= 2:
        cc = cross_check_documents(app_id, docs)
        if cc.hitl_required:
            hitl_reasons.extend([f"cross_check: {d}" for d in cc.discrepancies])

    # Collateral verification
    col = verify_collateral_chain(app_id, docs, PROPERTY_VALUATIONS)
    if col.hitl_required:
        hitl_reasons.extend([f"collateral: {i}" for i in (col.critical_issues + col.warnings)])

    # Overall quality
    avg_conf = sum(all_confidences) / len(all_confidences) if all_confidences else 0.0
    if avg_conf < 0.80:
        hitl_reasons.append(f"Overall quality too low: avg confidence={avg_conf:.2f}")

    # Determine final HITL level
    if docs_failed or (cc and cc.hitl_required) or (col.hitl_level == 2):
        final_hitl = HITLLevel.L2
    elif docs_flagged or (col.hitl_level == 1):
        final_hitl = HITLLevel.L1
    else:
        final_hitl = HITLLevel.NONE

    elapsed = time.time() - t0

    return DocumentIntelligenceResult(
        app_id=app_id, overall_quality=avg_conf,
        documents_ok=docs_ok, documents_flagged=docs_flagged, documents_failed=docs_failed,
        cross_check=cc, collateral=col,
        final_hitl_level=final_hitl, hitl_reasons=hitl_reasons,
        processing_time_s=round(elapsed, 3)
    )

# Run for all customers
for app_id in ["FC-2024-001", "FC-2024-003", "FC-2024-004"]:
    r = run_document_intelligence_pipeline(app_id)
    print(f"{'='*55}")
    print(f"  {app_id} — {APPLICATIONS[app_id]['name']}")
    print(f"  Quality:       {r.overall_quality:.2f}")
    print(f"  Docs OK:       {r.documents_ok}")
    print(f"  Docs Flagged:  {r.documents_flagged}")
    print(f"  Docs Failed:   {r.documents_failed}")
    print(f"  HITL Level:    {r.final_hitl_level.value}")
    if r.hitl_reasons:
        for reason in r.hitl_reasons[:3]:
            print(f"    → {reason}")
    print(f"  Time:          {r.processing_time_s}s")

## Section 6: Income Pattern Analysis

Phân tích pattern income từ monthly transaction data để detect volatility.

In [ ]:
import statistics

def analyze_income_pattern(monthly_inflows: List[float]) -> dict:
    """Analyze 12-month income pattern. Returns regularity classification."""
    if not monthly_inflows:
        return {"income_regularity": "unknown", "stable_income": 0, "variance_pct": None}

    avg    = statistics.mean(monthly_inflows)
    median = statistics.median(monthly_inflows)
    stdev  = statistics.stdev(monthly_inflows) if len(monthly_inflows) > 1 else 0
    cv     = stdev / avg if avg > 0 else 0  # coefficient of variation

    if cv < 0.10:
        regularity = "regular"     # < 10% variance → very stable
        stable_income = avg
    elif cv < 0.40:
        regularity = "irregular"   # 10-40% → some variation
        stable_income = median      # use median for robustness
    else:
        regularity = "volatile"    # > 40% → high risk
        stable_income = min(monthly_inflows)  # conservative: use minimum

    return {
        "income_regularity": regularity,
        "avg_income":        round(avg),
        "median_income":     round(median),
        "stable_income":     round(stable_income),  # used for loan capacity
        "variance_pct":      round(cv * 100, 1)
    }

# FC-2024-004: regular employment
fc4_monthly = [31_500_000, 31_200_000, 31_800_000, 31_000_000, 31_500_000,
               32_000_000, 31_200_000, 31_500_000, 31_800_000, 31_000_000,
               31_500_000, 31_200_000]
print("FC-2024-004 Income Pattern:")
print(analyze_income_pattern(fc4_monthly))
print()

# FC-2024-002: volatile income (simulated)
fc2_monthly = [12_000_000, 35_000_000, 8_000_000, 45_000_000, 15_000_000,
               28_000_000, 42_000_000, 9_000_000, 33_000_000, 17_000_000,
               38_000_000, 15_000_000]
print("FC-2024-002 Income Pattern (volatile):")
result_fc2 = analyze_income_pattern(fc2_monthly)
print(result_fc2)
print(f"→ Stable income for loan capacity: {result_fc2['stable_income']:,} VNĐ (min, conservative)")

## Section 7: 🎯 Practice — Document Tamper Detection

**Bài tập:** Implement `detect_tampering_signals()` function.

Một document bị tamper có nhiều signals:
1. **Round numbers:** net_income % 1_000_000 == 0 (round to millions)
2. **Impossible values:** net_income > gross_income, deductions = 0 with high income
3. **Unrealistic DTI:** if declared income >> bank statement by >30%
4. **Suspicious patterns:** gross == net (no tax on high income)

Implement function và test với FC-2024-003 data.

In [ ]:
def detect_tampering_signals(
    payslip: Dict[str, Any],
    bank_stmt: Dict[str, Any],
    declared_income: int
) -> Dict[str, Any]:
    """
    Detect document tampering signals.
    Returns: {
        'risk_level': 'LOW' | 'MEDIUM' | 'HIGH' | 'CRITICAL',
        'signals': List[str],
        'hitl_recommended': bool
    }

    TODO: Implement các checks sau:
    1. Round number check: net_income divisible by 1,000,000
    2. Impossible: gross == net (no deductions) for income > 20M VNĐ
    3. Impossible: deductions == 0 for income > 20M VNĐ
    4. High discrepancy: declared_income vs bank avg inflow > 30%
    5. Suspicious: payslip net vs bank avg differ > 20%

    Scoring:
    - 0 signals   → LOW
    - 1 signal    → MEDIUM
    - 2 signals   → HIGH
    - 3+ signals  → CRITICAL
    """
    signals = []

    # TODO: Implement checks here
    # 1. ...
    # 2. ...
    # 3. ...
    # 4. ...
    # 5. ...

    risk = ["LOW", "MEDIUM", "HIGH", "CRITICAL"][min(len(signals), 3)]
    return {
        "risk_level":         risk,
        "signals":            signals,
        "hitl_recommended":   risk in ("HIGH", "CRITICAL")
    }

# Test với FC-2024-003 (suspicious customer)
result = detect_tampering_signals(
    payslip=MOCK_DOCUMENTS["FC-2024-003"]["payslip"],
    bank_stmt={"avg_monthly_inflow": 35_000_000},  # actual bank shows only 35M not 120M
    declared_income=APPLICATIONS["FC-2024-003"]["declared_income"]
)

print("FC-2024-003 Tamper Detection:")
print(f"  risk_level:        {result['risk_level']}")
print(f"  signals:           {result['signals']}")
print(f"  hitl_recommended:  {result['hitl_recommended']}")
print()

# Test với FC-2024-004 (clean customer)
result_clean = detect_tampering_signals(
    payslip=MOCK_DOCUMENTS["FC-2024-004"]["payslip"],
    bank_stmt=MOCK_DOCUMENTS["FC-2024-004"]["bank_statement"],
    declared_income=APPLICATIONS["FC-2024-004"]["declared_income"]
)
print("FC-2024-004 Tamper Detection (should be LOW):")
print(f"  risk_level: {result_clean['risk_level']}")
print(f"  signals:    {result_clean['signals']}")

### 💡 Solution

<details>
<summary>Click để xem solution</summary>

```python
def detect_tampering_signals(payslip, bank_stmt, declared_income):
    signals = []
    net    = payslip.get('net_income', 0)
    gross  = payslip.get('gross_income', 0)
    deduct = payslip.get('total_deductions', -1)
    bank   = bank_stmt.get('avg_monthly_inflow', 0)

    # 1. Round number
    if net > 0 and net % 1_000_000 == 0:
        signals.append(f'Round number net_income={net:,} (millions exact)')

    # 2. Gross == Net for high income
    if gross > 20_000_000 and gross == net:
        signals.append(f'Gross == Net with high income — no deductions impossible')

    # 3. Zero deductions for high income
    if deduct == 0 and gross > 20_000_000:
        signals.append(f'Zero deductions on gross {gross:,} — impossible under VN tax law')

    # 4. Declared vs bank discrepancy > 30%
    if declared_income > 0 and bank > 0:
        disc = abs(declared_income - bank) / declared_income
        if disc > 0.30:
            signals.append(f'Income discrepancy {disc:.0%}: declared={declared_income:,} vs bank={bank:,}')

    # 5. Payslip net vs bank avg > 20%
    if net > 0 and bank > 0:
        disc = abs(net - bank) / net
        if disc > 0.20:
            signals.append(f'Payslip net {net:,} vs bank avg {bank:,} differ {disc:.0%}')

    risk = ['LOW', 'MEDIUM', 'HIGH', 'CRITICAL'][min(len(signals), 3)]
    return {'risk_level': risk, 'signals': signals, 'hitl_recommended': risk in ('HIGH', 'CRITICAL')}
```

FC-2024-003 nên detect CRITICAL (3+ signals): round_number + gross==net + zero_deductions + 70%+ discrepancy.
</details>